<a href="https://colab.research.google.com/github/Luispessoa18/hidrachat-image-worker/blob/master/HidraImg_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HidraImg Worker no Google Colab

Notebook pronto para instalar dependencias, baixar modelo Stable Diffusion e iniciar o worker HidraChat com GPU CUDA.

Antes de rodar: em `Runtime > Change runtime type`, selecione uma GPU.

In [ ]:
# 1. Conferir GPU
!nvidia-smi

Fri May 29 19:50:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   57C    P0             30W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# 2. Baixar o worker do GitHub
%cd /content
!rm -rf hidrachat-image-worker
!git clone https://github.com/Luispessoa18/hidrachat-image-worker
%cd /content/hidrachat-image-worker

/content
Cloning into 'hidrachat-image-worker'...
remote: Enumerating objects: 28, done.
remote: Counting objects: 100% (28/28), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 28 (delta 12), reused 27 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (28/28), 17.25 KiB | 5.75 MiB/s, done.
Resolving deltas: 100% (12/12), done.
/content/hidrachat-image-worker


In [ ]:
# 3. Instalar PyTorch CUDA + Diffusers
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q -r requirements.txt
!pip install -q xformers

## Configuracao

O email padrao ja esta definido como `luispessoa18@gmail.com`.

O modelo padrao abaixo e SD 1.5, mais leve para testar em T4. Para SDXL, troque `MODEL_ID` por `stabilityai/stable-diffusion-xl-base-1.0`.

In [ ]:
# 4 + 5. Definir email/modelo e baixar modelo corretamente
import os

EMAIL = "luispessoa18@gmail.com"

# MODELO SDXL LIGHTNING
MODEL_ID = "ByteDance/SDXL-Lightning"

# pasta local
LOCAL_MODEL_DIR = "/content/hidrachat-image-worker/models/sdxl-lightning"

# ENV
os.environ["HIDRACHAT_WORKER_EMAIL"] = EMAIL
os.environ["HIDRACHAT_MODEL_ID"] = LOCAL_MODEL_DIR
os.environ["HIDRACHAT_DEVICE"] = "cuda"
os.environ["HIDRACHAT_TORCH_DTYPE"] = "float16"
os.environ["HIDRACHAT_WORKER_NAME"] = "image-worker-colab"
os.environ["HIDRACHAT_REGION"] = "colab"

# IMPORTANTÍSSIMO:
# primeira vez precisa online
os.environ["HF_HUB_OFFLINE"] = "0"

# otimizações
os.environ["HIDRACHAT_LOCAL_FILES_ONLY"] = "0"
os.environ["HIDRACHAT_PRELOAD_MODEL"] = "1"
os.environ["HIDRACHAT_WARMUP_MODEL"] = "1"

print("Email:", EMAIL)
print("Modelo HF:", MODEL_ID)
print("Modelo local:", LOCAL_MODEL_DIR)

# baixar modelo
!python download_model.py \
  --model "stabilityai/stable-diffusion-xl-base-1.0" \
  --output "{LOCAL_MODEL_DIR}"

# conferir estrutura
!echo ""
!echo "==== Estrutura do modelo ===="
!ls -lah {LOCAL_MODEL_DIR}

# conferir arquivos essenciais
!test -f {LOCAL_MODEL_DIR}/model_index.json && echo "OK model_index.json" || echo "FALTOU model_index.json"
!test -d {LOCAL_MODEL_DIR}/unet && echo "OK unet/" || echo "FALTOU unet/"
!test -d {LOCAL_MODEL_DIR}/vae && echo "OK vae/" || echo "FALTOU vae/"
!test -d {LOCAL_MODEL_DIR}/text_encoder && echo "OK text_encoder/" || echo "FALTOU text_encoder/"
!test -d {LOCAL_MODEL_DIR}/tokenizer && echo "OK tokenizer/" || echo "FALTOU tokenizer/"

# agora ativa offline
os.environ["HF_HUB_OFFLINE"] = "1"

print("")
print("HF offline ativado.")

Email: luispessoa18@gmail.com
Modelo HF: ByteDance/SDXL-Lightning
Modelo local: /content/hidrachat-image-worker/models/sdxl-lightning
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
Fetching 18 files:   0% 0/18 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
Fetching 18 files: 100% 18/18 [01:58<00:00,  6.59s/it]
Download complete: 100% 13.9G/13.9G [01:58<00:00, 138MB/s]                Modelo baixado em: /content/hidrachat-image-worker/models/sdxl-lightning
Download complete: 100% 13.9G/13.9G [01:58<00:00, 117MB/s]

==== Estrutura do modelo ====
total 44K
drwxr-xr-x 10 root root 4.0K May 29 19:55 .
drwxr-xr-x  4 root root 4.0K May 29 19:53 ..
drwxr-xr-x  3 root root 4.0K

## Baixar modelo

Se o Hugging Face pedir aceite de licenca/token, rode a celula de login primeiro e cole seu token.

In [ ]:
# Opcional: login no Hugging Face, caso o modelo exija token
# from huggingface_hub import login
# login()

In [ ]:
# 5. Baixar somente os arquivos necessarios do modelo
!python download_model.py --model {MODEL_ID} --output {LOCAL_MODEL_DIR}

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
Fetching 0 files: 0it [00:00, ?it/s]
Download complete: : 0.00B [00:00, ?B/s]              Modelo baixado em: /content/hidrachat-image-worker/models/sdxl-lightning
Download complete: : 0.00B [00:00, ?B/s]


In [ ]:
# 6. Iniciar worker depois do modelo local estar pronto
!python colab_worker.py --model {LOCAL_MODEL_DIR}


Backend:      Diffusers/PyTorch (cuda)
Modelo:       /content/hidrachat-image-worker/models/sdxl-lightning
Servidor:     https://hidrachat.cloud

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Carregando Diffusers: /content/hidrachat-image-worker/models/sdxl-lightning
Keyword arguments {'safety_checker': None, 'requires_safety_checker': False} are not expected by StableDiffusionXLPipeline and will be ignored.
Loading pipeline components...:   0% 0/7 [00:00<?, ?it/s]
Loading weights:   0% 0/517 [00:00<?, ?it/s]
Loading weights:   0% 1/517 [00:00<00:00, 7096.96it/s, Materializing param=text_model.embeddings.position_embedding.weight]
Loading weights:   0% 1/517 [00:00<00:00, 2972.58it/s, Materializing param=text_model.embeddings.position_em

## Alternativa online, sem baixar modelo antes

Se quiser deixar o Diffusers baixar/cachear automaticamente durante o start, rode isto no lugar das celulas 4 e 5:

```python
%env HIDRACHAT_WORKER_EMAIL=luispessoa18@gmail.com
%env HIDRACHAT_MODEL_ID=runwayml/stable-diffusion-v1-5
%env HIDRACHAT_DEVICE=cuda
!python colab_worker.py --online-model
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 4. Definir email e modelo
import os

EMAIL = "luispessoa18@gmail.com"
MODEL_ID = "runwayml/stable-diffusion-v1-5"
LOCAL_MODEL_DIR = "/content/hidrachat-image-worker/models/sd15"

os.environ["HIDRACHAT_WORKER_EMAIL"] = EMAIL
os.environ["HIDRACHAT_MODEL_ID"] = LOCAL_MODEL_DIR
os.environ["HIDRACHAT_DEVICE"] = "cuda"
os.environ["HIDRACHAT_TORCH_DTYPE"] = "auto"
os.environ["HIDRACHAT_WORKER_NAME"] = "image-worker-colab"
os.environ["HIDRACHAT_REGION"] = "colab"
os.environ["HIDRACHAT_LOCAL_FILES_ONLY"] = "1"
os.environ["HIDRACHAT_PRELOAD_MODEL"] = "1"
os.environ["HIDRACHAT_WARMUP_MODEL"] = "1"

print("Email:", os.environ["HIDRACHAT_WORKER_EMAIL"])
print("Modelo local:", os.environ["HIDRACHAT_MODEL_ID"])

Email: luispessoa18@gmail.com
Modelo local: /content/hidrachat-image-worker/models/sd15
